# 04 — The Folding Step

This is the **heart of Nova**: given two Relaxed R1CS instances, fold them into one.

## The Algorithm (NIFS — Non-Interactive Folding Scheme)

Given $(\text{inst}_1, \text{wit}_1)$ and $(\text{inst}_2, \text{wit}_2)$:

1. **Compute cross-term:** $\mathbf{T} = A\mathbf{z}_1 \circ B\mathbf{z}_2 + A\mathbf{z}_2 \circ B\mathbf{z}_1 - u_1 C\mathbf{z}_2 - u_2 C\mathbf{z}_1$
2. **Commit:** Send $\text{com}_T = \text{Com}(\mathbf{T})$ to verifier
3. **Challenge:** $r \leftarrow \text{Fiat-Shamir}(\text{com}_T, \text{inst}_1, \text{inst}_2)$
4. **Fold:**
   - $\mathbf{E}' = \mathbf{E}_1 + r \cdot \mathbf{T} + r^2 \cdot \mathbf{E}_2$
   - $u' = u_1 + r \cdot u_2$
   - $\mathbf{x}' = \mathbf{x}_1 + r \cdot \mathbf{x}_2$
   - $\mathbf{W}' = \mathbf{W}_1 + r \cdot \mathbf{W}_2$

**Cost: one commitment (com_T) per fold.** That's it.

**Nova paper reference:** Section 4, Construction 4

In [ ]:
import numpy as np
from nano_nova.field import GF
from nano_nova.commitment import commit
from nano_nova.examples import fibonacci_step_circuit, fibonacci_step_fn, fibonacci_witness_fn
from nano_nova.r1cs import r1cs_to_relaxed, trivial_relaxed
from nano_nova.folding import compute_cross_term, fold

shape = fibonacci_step_circuit()

In [ ]:
# Create two Fibonacci step instances
z1 = GF(np.array([1, 1]))  # (1,1) -> (1,2)
z1_out = fibonacci_step_fn(z1)
w1 = fibonacci_witness_fn(z1)
x1 = GF(np.concatenate([z1, z1_out]))
inst1, wit1 = r1cs_to_relaxed(shape, x1, w1, commit)

z2 = GF(np.array([1, 2]))  # (1,2) -> (2,3)
z2_out = fibonacci_step_fn(z2)
w2 = fibonacci_witness_fn(z2)
x2 = GF(np.concatenate([z2, z2_out]))
inst2, wit2 = r1cs_to_relaxed(shape, x2, w2, commit)

print("Instance 1: (1,1) → (1,2)")
print(f"  u={int(inst1.u)}, x={[int(v) for v in inst1.x]}, W={[int(v) for v in wit1.W]}")
print(f"  E={[int(v) for v in wit1.E]}")

print("\nInstance 2: (1,2) → (2,3)")
print(f"  u={int(inst2.u)}, x={[int(v) for v in inst2.x]}, W={[int(v) for v in wit2.W]}")
print(f"  E={[int(v) for v in wit2.E]}")

## Step 1: Compute the Cross-Term T

$\mathbf{T} = A\mathbf{z}_1 \circ B\mathbf{z}_2 + A\mathbf{z}_2 \circ B\mathbf{z}_1 - u_1 C\mathbf{z}_2 - u_2 C\mathbf{z}_1$

This is the "interference" — the part that would break naive folding. We commit to it so the verifier can check, then absorb it into $\mathbf{E}$.

In [ ]:
# Step 1: Compute cross-term
T = compute_cross_term(shape, inst1, wit1, inst2, wit2)
print(f"Cross-term T = {[int(v) for v in T]}")

# Let's also see the components
z1_full = shape._make_relaxed_z(inst1.u, inst1.x, wit1.W)
z2_full = shape._make_relaxed_z(inst2.u, inst2.x, wit2.W)

Az1 = shape.A @ z1_full
Bz1 = shape.B @ z1_full
Az2 = shape.A @ z2_full
Bz2 = shape.B @ z2_full
Cz1 = shape.C @ z1_full
Cz2 = shape.C @ z2_full

print(f"\nComponents:")
print(f"  Az1∘Bz2 = {Az1 * Bz2}")
print(f"  Az2∘Bz1 = {Az2 * Bz1}")
print(f"  u1·Cz2  = {inst1.u * Cz2}")
print(f"  u2·Cz1  = {inst2.u * Cz1}")
print(f"  T = sum  = {Az1*Bz2 + Az2*Bz1 - inst1.u*Cz2 - inst2.u*Cz1}")

## Steps 2-4: Commit, Challenge, Fold

In [ ]:
# Step 2: Commit to T
com_T = commit(T)
print(f"Step 2: com_T = {com_T}")

# Step 3: Fiat-Shamir challenge (hash of transcript)
from nano_nova.folding import fiat_shamir_challenge
r = fiat_shamir_challenge(com_T, inst1, inst2)
print(f"\nStep 3: challenge r = {int(r)}")

# Step 4: Fold everything
r_sq = r * r
E_folded = wit1.E + r * T + r_sq * wit2.E
u_folded = inst1.u + r * inst2.u
x_folded = inst1.x + r * inst2.x
W_folded = wit1.W + r * wit2.W

print(f"\nStep 4 — Folded instance:")
print(f"  u' = u1 + r*u2 = {int(inst1.u)} + {int(r)}*{int(inst2.u)} = {int(u_folded)}")
print(f"  E' = E1 + r*T + r²*E2 = {[int(v) for v in E_folded]}")
print(f"  x' = x1 + r*x2 = {[int(v) for v in x_folded]}")
print(f"  W' = W1 + r*W2 = {[int(v) for v in W_folded]}")

In [ ]:
# Verify: does the folded instance satisfy Relaxed R1CS?
from nano_nova.r1cs import RelaxedR1CSInstance, RelaxedR1CSWitness

folded_inst = RelaxedR1CSInstance(
    com_E=commit(E_folded), u=u_folded, x=x_folded, com_W=commit(W_folded)
)
folded_wit = RelaxedR1CSWitness(E=E_folded, W=W_folded)

satisfied = shape.is_relaxed_satisfied(folded_inst, folded_wit)
print(f"Folded instance satisfies Relaxed R1CS: {satisfied}")

# Let's verify the math: Az'∘Bz' should equal u'·Cz' + E'
z_folded = shape._make_relaxed_z(u_folded, x_folded, W_folded)
Az = shape.A @ z_folded
Bz = shape.B @ z_folded
Cz = shape.C @ z_folded
lhs = Az * Bz
rhs = u_folded * Cz + E_folded

print(f"\nLHS (Az'∘Bz') = {[int(v) for v in lhs]}")
print(f"RHS (u'Cz'+E') = {[int(v) for v in rhs]}")
print(f"Match: {np.all(lhs == rhs)} ✓")

## Using the `fold()` Function

The above was step-by-step for understanding. In practice, we use the `fold()` function:

In [ ]:
# Same thing in one call
folded_inst2, folded_wit2 = fold(shape, inst1, wit1, inst2, wit2)
print(f"fold() result satisfies Relaxed R1CS: {shape.is_relaxed_satisfied(folded_inst2, folded_wit2)}")

## Chaining Folds

The magic: the output of a fold is itself a valid Relaxed R1CS instance, so we can fold again. The error vector $\mathbf{E}$ accumulates cross-terms from each fold, but the instance stays the same size.

In [ ]:
# Chain 5 folds, starting with trivial accumulator
acc_inst, acc_wit = trivial_relaxed(shape, commit)
z = GF(np.array([1, 1]))

print(f"{'Step':>4} | {'z':>12} | {'u':>6} | {'E norm':>10} | {'Valid':>5}")
print("-" * 55)

for i in range(5):
    z_next = fibonacci_step_fn(z)
    w = fibonacci_witness_fn(z)
    x = GF(np.concatenate([z, z_next]))
    fresh_inst, fresh_wit = r1cs_to_relaxed(shape, x, w, commit)
    
    acc_inst, acc_wit = fold(shape, acc_inst, acc_wit, fresh_inst, fresh_wit)
    valid = shape.is_relaxed_satisfied(acc_inst, acc_wit)
    
    # E "norm" (sum of absolute values, for visualization)
    e_norm = sum(min(int(v), int(GF(0) - v)) for v in acc_wit.E)
    
    print(f"{i+1:>4} | ({int(z_next[0]):>4},{int(z_next[1]):>4}) | {int(acc_inst.u):>6} | {e_norm:>10} | {'✓' if valid else '✗':>5}")
    z = z_next

print(f"\nAfter 5 folds: instance size unchanged, all folds valid.")
print(f"E grows (accumulating cross-terms), but the structure is the same.")

## Why This Works: The Proof

The correctness of folding follows from expanding the quadratic and matching terms:

$$A\mathbf{z}' \circ B\mathbf{z}' = A(\mathbf{z}_1 + r\mathbf{z}_2) \circ B(\mathbf{z}_1 + r\mathbf{z}_2)$$

$$= \underbrace{A\mathbf{z}_1 \circ B\mathbf{z}_1}_{= u_1 C\mathbf{z}_1 + \mathbf{E}_1} + r \cdot \underbrace{(A\mathbf{z}_1 \circ B\mathbf{z}_2 + A\mathbf{z}_2 \circ B\mathbf{z}_1)}_{\text{includes } \mathbf{T}} + r^2 \cdot \underbrace{A\mathbf{z}_2 \circ B\mathbf{z}_2}_{= u_2 C\mathbf{z}_2 + \mathbf{E}_2}$$

Meanwhile: $u' C\mathbf{z}' + \mathbf{E}' = (u_1 + ru_2) C(\mathbf{z}_1 + r\mathbf{z}_2) + \mathbf{E}_1 + r\mathbf{T} + r^2\mathbf{E}_2$

Expanding and using $\mathbf{T} = A\mathbf{z}_1 \circ B\mathbf{z}_2 + A\mathbf{z}_2 \circ B\mathbf{z}_1 - u_1 C\mathbf{z}_2 - u_2 C\mathbf{z}_1$, everything cancels. ∎

## Key Takeaway

Nova's folding:
- **Combines** two instances into one of the **same size**
- **Cost per fold:** one commitment (com_T) + field operations
- **Error $\mathbf{E}$ accumulates** but the structure is preserved
- The cross-term $\mathbf{T}$ plays the role of a **renormalization counter-term**

Next: [05_ivc_demo.ipynb](05_ivc_demo.ipynb) — putting it all together for Incrementally Verifiable Computation.